# Customer Support Triage Pipeline

## Objective

Build a multi-agent workflow that:

1. Classifies customer intent and priority
2. Retrieves relevant knowledge base information
3. Generates a customer-facing response

## Agents

### Classifier Agent
Identifies:
- Intent
- Priority

### Retrieval Agent
Fetches relevant KB articles.

### Composer Agent
Creates a professional response using:
- Customer query
- Retrieved KB information

## Workflow

Customer Query
       ↓
Classifier Agent
       ↓
Retrieval Agent
       ↓
Composer Agent
       ↓
Final Response

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    base_url="https://llmgw-wp.tekstac.com",
    temperature=0
)

In [2]:
knowledge_base = {
    "billing": [
        "Customers can download invoices from the Billing section.",
        "Refund requests are processed within 5-7 business days.",
        "Failed payments can be retried from the payment dashboard."
    ],

    "technical issue": [
        "Restarting the application solves most session issues.",
        "Clear browser cache if login pages fail to load.",
        "Check internet connectivity before troubleshooting further."
    ],

    "account access": [
        "Password reset emails are sent instantly.",
        "Accounts may be temporarily locked after multiple failed login attempts.",
        "Users can unlock accounts via the self-service portal."
    ]
}

In [3]:
class SupportState(TypedDict):
    query: str
    intent: str
    priority: str
    kb_results: list
    draft_response: str
    final_response: str

In [4]:
def classifier_agent(state: SupportState):

    prompt = f"""
    Classify the customer request.

    Query:
    {state['query']}

    Return:

    Intent: Billing / Technical Issue / Account Access
    Priority: High / Medium / Low
    """

    result = llm.invoke(prompt).content

    intent = "billing"

    if "technical" in result.lower():
        intent = "technical issue"

    elif "account" in result.lower():
        intent = "account access"

    priority = "Medium"

    if "high" in result.lower():
        priority = "High"

    elif "low" in result.lower():
        priority = "Low"

    return {
        "intent": intent,
        "priority": priority
    }

In [5]:
def retrieval_agent(state: SupportState):

    passages = knowledge_base.get(
        state["intent"],
        []
    )

    return {
        "kb_results": passages[:3]
    }

In [6]:
def composer_agent(state: SupportState):

    prompt = f"""
    Customer Query:
    {state['query']}

    Knowledge Base:
    {state['kb_results']}

    Generate a professional customer response.
    """

    response = llm.invoke(prompt).content

    return {
        "draft_response": response,
        "final_response": response
    }

In [7]:
builder = StateGraph(SupportState)

builder.add_node("classifier", classifier_agent)
builder.add_node("retrieval", retrieval_agent)
builder.add_node("composer", composer_agent)

builder.add_edge(START, "classifier")
builder.add_edge("classifier", "retrieval")
builder.add_edge("retrieval", "composer")
builder.add_edge("composer", END)

graph = builder.compile()

In [8]:
queries = [
    "I was charged twice for my subscription.",
    "The application crashes every time I login.",
    "I am unable to reset my password.",
    "My payment failed even though my card is valid.",
    "My account is locked after several login attempts."
]

for q in queries:

    result = graph.invoke({
        "query": q
    })

    print("=" * 80)
    print("CUSTOMER QUERY")
    print(q)

    print("\nCLASSIFICATION")
    print("Intent :", result["intent"])
    print("Priority :", result["priority"])

    print("\nRETRIEVED KB PASSAGES")
    for item in result["kb_results"]:
        print("-", item)

    print("\nFINAL RESPONSE")
    print(result["final_response"])

    print()

CUSTOMER QUERY
I was charged twice for my subscription.

CLASSIFICATION
Intent : billing
Priority : High

RETRIEVED KB PASSAGES
- Customers can download invoices from the Billing section.
- Refund requests are processed within 5-7 business days.
- Failed payments can be retried from the payment dashboard.

FINAL RESPONSE
# Customer Response

Thank you for bringing this to our attention. We sincerely apologize for the duplicate charge on your subscription.

**Here's what we recommend:**

1. **Verify the charges** – Please download your invoices from the Billing section to confirm the duplicate transaction and gather the details.

2. **Submit a refund request** – Once you've confirmed the duplicate charge, please contact our support team with your invoice details. Refund requests are typically processed within 5-7 business days.

3. **Prevent future issues** – If you notice any failed payment attempts in your payment dashboard, you can retry them from there to avoid accidental duplicate 

# Meeting Transcript Processing Pipeline

## Objective

Convert a meeting transcript into:

1. Segmented discussion sections
2. Meeting summary
3. Action items

## Workflow

Transcript
    ↓
Segmentation Agent
    ↓
Summarizer Agent
    ↓
Action Extraction Agent
    ↓
Summary + Action List

In [9]:
class MeetingState(TypedDict):
    transcript: str
    segments: str
    summary: str
    action_items: str

In [10]:
def segmentation_agent(state: MeetingState):

    prompt = f"""
    Divide the meeting transcript into logical discussion sections.

    Transcript:
    {state['transcript']}
    """

    segments = llm.invoke(prompt).content

    return {
        "segments": segments
    }

In [11]:
def summarizer_agent(state: MeetingState):

    prompt = f"""
    Summarize the transcript sections.

    Content:
    {state['segments']}

    Create a concise 3-4 sentence meeting summary.
    """

    summary = llm.invoke(prompt).content

    return {
        "summary": summary
    }

In [12]:
def action_agent(state: MeetingState):

    prompt = f"""
    Extract action items.

    Content:
    {state['segments']}

    Return:

    Action
    Owner
    Due Date
    Status
    """

    actions = llm.invoke(prompt).content

    return {
        "action_items": actions
    }

In [14]:
builder = StateGraph(MeetingState)

builder.add_node("segment", segmentation_agent)
builder.add_node("summarize", summarizer_agent)
builder.add_node("actions", action_agent)

builder.add_edge(START, "segment")
builder.add_edge("segment", "summarize")
builder.add_edge("summarize", "actions")
builder.add_edge("actions", END)

meeting_graph = builder.compile()

In [15]:
transcript = """
Rahul: Let's finalize the customer portal deployment.

Santosh: Infrastructure setup is completed.

Rahul: Testing still needs to be finished before release.

Ananya: I'll complete testing by July 10.

Santosh: Security review is required before production deployment.

Rahul: Agreed. Security review should finish by July 12.

Ananya: Once testing and security review are done,
we can release the portal.
"""

In [16]:
result = meeting_graph.invoke(
    {
        "transcript": transcript
    }
)

print("=" * 80)
print("MEETING SUMMARY")
print(result["summary"])

print("\n" + "=" * 80)
print("ACTION ITEMS")
print(result["action_items"])

MEETING SUMMARY
# Meeting Summary

The team discussed finalizing the customer portal deployment with infrastructure setup already completed. Testing and security review were identified as critical blocking items, with Ananya committing to complete testing by July 10 and Santosh scheduling the security review for completion by July 12. Once both approvals are obtained, the portal will be ready for production release.

ACTION ITEMS
# Action Items

| Action | Owner | Due Date | Status |
|--------|-------|----------|--------|
| Complete testing for customer portal deployment | Ananya | July 10 | Pending |
| Conduct security review before production deployment | Santosh | July 12 | Pending |
| Finalize customer portal deployment | Rahul | After July 12 | Pending |
